In [1]:
import wrds
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.regression.rolling import RollingOLS
import matplotlib.pyplot as plt

/Users/ywo/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# author's datashare
df = pd.read_csv('/Users/ywo/Downloads/Industry Project/datashare (1)/datashare.csv')
df['DATE']=pd.to_datetime(df['DATE'], format='%Y%m%d')

In [3]:
db = wrds.Connection()

Enter your WRDS username [ywo]:ywhan
Enter your password:········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [4]:
# df['date_str'] = pd.to_datetime(df['DATE']).dt.strftime('%Y-%m-%d')
permnos = df['permno'].unique().tolist()
dates = df['DATE'].unique().tolist()

query = f"""
SELECT 
    b.permno, 
    b.date, 
    b.ret, 
    c.rf,
    (b.ret - c.rf) AS exret
FROM crsp.dsf b
LEFT JOIN ff.factors_daily c ON b.date = c.date
WHERE b.permno IN ({','.join(map(str, permnos))})
  AND b.date IN ({','.join([f"'{d}'" for d in dates])})
"""


df_dsf_raw = db.raw_sql(query)
df_dsf_raw['date'] = pd.to_datetime(df_dsf_raw['date'])

df_exret = pd.merge(
    df, 
    df_dsf_raw, 
    left_on=['permno', 'DATE'], 
    right_on=['permno', 'date'], 
    how='left'
)

print(f"success: {df_exret['exret'].notna().sum()}")

success: 4096711


In [5]:
df_exret.drop(columns=['date'])

,permno,DATE,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,...,ill,maxret,retvol,std_dolvol,std_turn,zerotrade,sic2,ret,rf,exret
0,10006,1957-01-31,8.224900e+04,1.122846,1.260784,0.047180,9.569953,0.025742,0.046433,0.044843,...,9.411565e-08,0.015453,0.008058,0.355638,0.460420,1.120996e-07,37.0,-0.008,0.0001,-0.0081
1,10014,1957-01-31,3.903375e+03,0.426734,0.182102,-0.275641,6.237836,0.072103,0.046433,-0.086957,...,6.610609e-06,0.047619,0.033495,1.152126,1.169610,9.229146e-08,NaN,-0.041667,0.0001,-0.041767
2,10022,1957-01-31,9.273250e+03,1.066449,1.137313,-0.025490,7.008844,0.027648,0.046433,-0.060377,...,2.286832e-06,0.020833,0.015589,0.815777,0.679803,1.181757e-07,NaN,0.0,0.0001,-0.0001
3,10030,1957-01-31,5.446588e+04,0.926038,0.857547,0.018171,9.825337,0.021700,0.046433,0.044633,...,1.464273e-07,0.039326,0.015849,0.739302,1.333656,6.126699e-08,NaN,0.0,0.0001,-0.0001
4,10057,1957-01-31,4.025000e+04,1.247748,1.556875,0.025785,7.901007,0.025506,0.046433,0.086667,...,1.380375e-06,0.056856,0.019945,0.755510,0.410391,3.315790e+00,NaN,-0.005093,0.0001,-0.005193
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4117295,93423,2021-12-31,3.144398e+06,1.923076,3.698221,-0.673385,16.305232,0.062619,0.491455,-0.110868,...,3.698841e-10,0.073435,0.029678,0.399725,9.215947,5.031856e-09,79.0,0.007096,0.0,0.007096
4117296,93426,2021-12-31,4.326610e+05,1.224523,1.499457,-0.061462,12.419552,0.033723,0.346308,0.007040,...,1.097624e-08,0.046055,0.019302,0.481331,1.649750,3.023395e-08,36.0,-0.00349,0.0,-0.00349
4117297,93427,2021-12-31,4.092710e+06,0.887083,0.786917,-0.080295,14.884834,0.045395,0.346308,0.151667,...,8.593845e-10,0.131563,0.032861,0.440057,2.444487,2.224496e-08,36.0,-0.003113,0.0,-0.003113
4117298,93434,2021-12-31,1.130478e+05,0.512942,0.263110,-0.543760,12.243872,0.088124,0.270005,-0.319347,...,1.145228e-07,0.036932,0.035858,0.968425,4.557546,3.155353e-08,1.0,0.011111,0.0,0.011111


In [ ]:
# len(df_exret['sic2'].unique())

In [6]:
# 8 macro variables from websites
df_macro=pd.read_excel('/Users/ywo/Downloads/Industry Project/datashare (1)/Data2024.xlsx', sheet_name='Monthly')
macro=['tbl','d/p','e/p','b/m','tms','dfy','ntis','svar']
df_macro[macro]=df_macro[macro].shift(1)


In [7]:
df_macro['date'] = pd.to_datetime(df_macro['yyyymm'], format='%Y%m')
df_macro['year_month'] = df_macro['date'].dt.to_period('M')


In [8]:
df_macro=df_macro[['yyyymm','tbl','d/p','e/p','b/m','tms','dfy','ntis','svar','year_month']]
# df_macro

In [9]:
df_exret['year_month'] = df_exret['DATE'].dt.to_period('M')
df_exret = df_exret[df_exret['ret'].notna()]
df_exret = df_exret.reset_index(drop=True)
# df_exret

In [10]:
features=list(df_exret.columns)[2:96]
# features

In [11]:
def norm_rank(data):
    ranks=data.rank(method='average',na_option='keep')
    n=ranks.count()
    mapped = (ranks / (n + 1)) * 2 - 1
    return mapped.fillna(0)
df_exret[features]=df_exret.groupby('DATE')[features].apply(norm_rank).reset_index(drop=True)
# df_exret

In [13]:
# final data
df_merged=pd.merge(df_exret,df_macro,on='year_month',how='left')
df_merged.drop(columns=['yyyymm','year_month'])

,permno,DATE,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,...,rf,exret,tbl,d/p,e/p,b/m,tms,dfy,ntis,svar
0,10006,1957-01-31,0.274976,0.175899,0.175899,0.340446,0.465138,-0.241983,0.000000,0.397901,...,0.0001,-0.0081,0.0321,0.037283,0.073066,0.544177,0.0024,0.0062,0.02615,0.001020
1,10014,1957-01-31,-0.939106,-0.803693,-0.803693,-0.788555,-0.866285,0.988338,0.000000,-0.949427,...,0.0001,-0.041767,0.0321,0.037283,0.073066,0.544177,0.0024,0.0062,0.02615,0.001020
2,10022,1957-01-31,-0.741199,0.094266,0.094266,-0.030068,-0.688634,-0.074830,0.000000,-0.883588,...,0.0001,-0.0001,0.0321,0.037283,0.073066,0.544177,0.0024,0.0062,0.02615,0.001020
3,10030,1957-01-31,0.059943,-0.137026,-0.137026,0.206596,0.574021,-0.607386,0.000000,0.395038,...,0.0001,-0.0001,0.0321,0.037283,0.073066,0.544177,0.0024,0.0062,0.02615,0.001020
4,10057,1957-01-31,-0.090390,0.393586,0.393586,0.245393,-0.379179,-0.271137,0.000000,0.692748,...,0.0001,-0.005193,0.0321,0.037283,0.073066,0.544177,0.0024,0.0062,0.02615,0.001020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4096706,93423,2021-12-31,0.522734,0.821825,0.820751,-0.663066,0.707389,0.194633,0.823155,-0.473839,...,0.0,0.007096,0.0005,0.013141,0.041684,0.185394,0.0151,0.0066,0.01564,0.001327
4096707,93426,2021-12-31,-0.109470,0.240787,0.237209,0.472125,-0.294672,-0.503399,0.305499,0.535567,...,0.0,-0.00349,0.0005,0.013141,0.041684,0.185394,0.0151,0.0066,0.01564,0.001327
4096708,93427,2021-12-31,0.587895,-0.266547,-0.271199,0.423693,0.375626,-0.141682,0.305499,0.908583,...,0.0,-0.003113,0.0005,0.013141,0.041684,0.185394,0.0151,0.0066,0.01564,0.001327
4096709,93434,2021-12-31,-0.624095,-0.747048,-0.752773,-0.534843,-0.347954,0.516995,0.220984,-0.930335,...,0.0,0.011111,0.0005,0.013141,0.041684,0.185394,0.0151,0.0066,0.01564,0.001327


In [ ]:
# mat_stock = df_merged[features].values
# mat_macro = df_merged[macro].values
# num_rows = len(df_merged)
# num_inter_cols = len(features) * len(macro)
# inter_matrix = np.zeros((num_rows, num_inter_cols), dtype='float32') 

# col_names = []
# current_col = 0

# for m_col_idx, m_col_name in enumerate(macro):
#     start = current_col
#     end = current_col + len(features)
    
#     inter_matrix[:, start:end] = mat_stock * mat_macro[:, [m_col_idx]]
    
#     col_names.extend([f"{s}_{m_col_name}" for s in features])
#     current_col = end

# df_inter = pd.DataFrame(inter_matrix, columns=col_names, index=df_merged.index)
# df_final = pd.concat([df_merged, df_inter], axis=1)

# del inter_matrix, df_inter
# import gc
# gc.collect()
